In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

2025-07-02 05:26:44.063536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751434004.255470      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751434004.310226      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
# Change model to BERTweet
MODEL_NAME = "sarkerlab/SocBERT-base"
TRAIN_PATH = "/kaggle/input/qna-task2/qna_train.csv"
TEST_PATH = "/kaggle/input/qna-task2/qna_test.csv"
GAMMA = 2.0
SEED = 42

# Set seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# Load data
df = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

df = df.dropna(subset=["MAIN", "comment_body", "relevance"])
df["MAIN"] = df["MAIN"].astype(str)
df["comment_body"] = df["comment_body"].astype(str)
df["text_pair"] = list(zip(df["MAIN"], df["comment_body"]))

# Train/Val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text_pair"].tolist(), df["relevance"].tolist(), 
    test_size=0.1, stratify=df["relevance"], random_state=SEED
)

# BERTweet tokenizer (note: BERTweet uses a different tokenizer than standard BERT)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Dataset Class (same as before)
class RedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        topic, comment = self.texts[idx]
        encoding = self.tokenizer(
            topic,
            comment,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Prepare Datasets
train_dataset = RedditDataset(train_texts, train_labels, tokenizer)
val_dataset = RedditDataset(val_texts, val_labels, tokenizer)

# Focal Loss Trainer (same as before)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.15, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

class FocalLossTrainer(Trainer):
    def __init__(self, *args, alpha=0.15, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = FocalLoss(alpha=alpha, gamma=gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Load BERTweet model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Metrics (same as before)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

# Training Arguments (adjusted for BERTweet)
training_args = TrainingArguments(
    output_dir="./bertweet_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
)

# Initialize Trainer
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    alpha=0.15,
    gamma=2.0
)

# Train
trainer.train()

# Evaluate
trainer.evaluate()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at sarkerlab/SocBERT-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_35/2608346819.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty e

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.013400,0.012269,0.886643,0.434211,0.811475,0.296407
2,0.013000,0.012341,0.887083,0.430155,0.829060,0.290419
3,0.011900,0.012295,0.885764,0.469388,0.737179,0.344311
4,0.010100,0.012726,0.877856,0.494545,0.629630,0.407186
5,0.009200,0.013506,0.880053,0.489720,0.651741,0.392216


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

{'eval_loss': 0.012726282700896263,
 'eval_accuracy': 0.8778558875219684,
 'eval_f1': 0.4945454545454545,
 'eval_precision': 0.6296296296296297,
 'eval_recall': 0.40718562874251496,
 'eval_runtime': 40.7552,
 'eval_samples_per_second': 55.846,
 'eval_steps_per_second': 0.883,
 'epoch': 5.0}

In [8]:
df_test["MAIN"] = df_test["MAIN"].astype(str)
df_test["comment_body"] = df_test["comment_body"].astype(str)
df_test_texts = list(zip(df_test["MAIN"], df_test["comment_body"]))
df_test_labels = df_test["relevance"].tolist()

test_dataset = RedditDataset(df_test_texts, df_test_labels, tokenizer)

predictions = trainer.predict(test_dataset)
test_preds = np.argmax(predictions.predictions, axis=1)

print("Predictions on test set complete.")
test_true_labels = predictions.label_ids
test_accuracy = accuracy_score(test_true_labels, test_preds)
test_f1 = f1_score(test_true_labels, test_preds)
test_precision = precision_score(test_true_labels, test_preds)
test_recall = recall_score(test_true_labels, test_preds)

print("\nTest Set Metrics:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Predictions on test set complete.

Test Set Metrics:
Accuracy: 0.8782
F1 Score: 0.4671
Precision: 0.6522
Recall: 0.3639


In [9]:
class RedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels  # labels can now be None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        topic, comment = self.texts[idx]
        encoding = self.tokenizer(
            topic,
            comment,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        
        # --- Add this conditional check ---
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        # If self.labels is None, we simply don't add the "labels" key to the item
        return item

df_test_submission = pd.read_csv("/kaggle/input/qna-task2/CRYPTO_QnA_TEST.csv")
df_test_submission["MAIN"] = df_test_submission["MAIN"].astype(str)
df_test_submission["comment_body"] = df_test_submission["comment_body"].astype(str)
test_submission_texts = list(zip(df_test_submission["MAIN"], df_test_submission["comment_body"]))
test_submission_dataset = RedditDataset(test_submission_texts, None, tokenizer)

submission_predictions = trainer.predict(test_submission_dataset)
final_predicted_relevance = np.argmax(submission_predictions.predictions, axis=1)

df_test_submission["relevance"] = final_predicted_relevance

final_submission_df = df_test_submission[["title", "selftext", "MAIN", "comment_body", "relevance"]]
submission_filename = "crypto_test_qna.csv"
final_submission_df.to_csv(submission_filename, index=False)
print(f"\nSubmission file '{submission_filename}' created successfully!")
print(f"Shape of submission file: {final_submission_df.shape}")
print(final_submission_df.head())

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai


Submission file 'crypto_test_qna.csv' created successfully!
Shape of submission file: (6323, 5)
                                               title  \
0  Which exchange to use to see holdings increase...   
1  The end of this year is approaching, how would...   
2                        When do you pull out? (Ha!)   
3  ETH, BTC, ADA, ATOM, ALGO, any other promising...   
4                  Convince me any of this has value   

                                            selftext  \
0  Wazirx doesn't show how much a portfolio has c...   
1  As the end of the year approaches, I am intere...   
2  So I promised my SO that I would only invest a...   
3  These so far are the ones I’ve locked in and c...   
4  I’ve been watching crypto from the outside for...   

                                                MAIN  \
0  Which exchange to use to see holdings increase...   
1  the end of this year is approaching, how would...   
2  when do you pull out? (ha!) so i promised my s...   
3  et

In [11]:
checkpoint_folder = "/kaggle/working/bertweet_results/checkpoint-3205"

zip_filename = "task2-soc-bert.zip"

print(f"\nZipping the checkpoint folder '{checkpoint_folder}' into '{zip_filename}'...")
!zip -r {zip_filename} {checkpoint_folder}

from IPython.display import FileLink
print(f"\nClick the link below to download your checkpoints:")
FileLink(zip_filename)


Zipping the checkpoint folder '/kaggle/working/bertweet_results/checkpoint-3205' into 'task2-soc-bert.zip'...
  adding: kaggle/working/bertweet_results/checkpoint-3205/ (stored 0%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/merges.txt (deflated 56%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/scheduler.pt (deflated 56%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/optimizer.pt (deflated 35%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/config.json (deflated 49%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/model.safetensors (deflated 7%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/vocab.json (deflated 69%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/special_tokens_map.json (deflated 85%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/tokenizer_config.json (deflated 76%)
  adding: kaggle/working/bertweet_results/checkpoint-3205/training_args.bin (deflated 51%)
  adding: kaggle/wor

/kaggle/working/task2-soc-bert.zip